In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!cp "/content/drive/MyDrive/NewDataset/image.zip" /content/
!cp "/content/drive/MyDrive/NewDataset/ClothSegmentation.zip" /content/

In [ ]:
!unzip /content/image.zip -d /content/images_folder/
!unzip /content/ClothSegmentation.zip -d /content/cloth_segmentation_folder/

Archive:  /content/image.zip
replace /content/images_folder/image/00050_00.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: /content/images_folder/image/00050_00.jpg  
  inflating: /content/images_folder/image/00051_00.jpg  
  inflating: /content/images_folder/image/00052_00.jpg  
  inflating: /content/images_folder/image/00053_00.jpg  
  inflating: /content/images_folder/image/00054_00.jpg  
  inflating: /content/images_folder/image/00056_00.jpg  
  inflating: /content/images_folder/image/00057_00.jpg  
  inflating: /content/images_folder/image/00059_00.jpg  
  inflating: /content/images_folder/image/00062_00.jpg  
  inflating: /content/images_folder/image/00066_00.jpg  
  inflating: /content/images_folder/image/00068_00.jpg  
  inflating: /content/images_folder/image/00071_00.jpg  
  inflating: /content/images_folder/image/00074_00.jpg  
  inflating: /content/images_folder/image/00076_00.jpg  
  inflating: /content/images_folder/image/00077_00.jpg  
  inflating: /content/ima

In [ ]:
# @title Setup Environment and Download Resources
%cd /content/
!rm -rf cloth-segmentation
!git clone https://github.com/ozzy404/cloth-segmentation.git
%cd cloth-segmentation
!gdown --id 1mhF3yqd7R-Uje092eypktNl-RoZNuiCJ
!mkdir input_images
!mkdir output_images

/content
Cloning into 'cloth-segmentation'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 157 (delta 26), reused 6 (delta 5), pack-reused 112 (from 1)
Receiving objects: 100% (157/157), 16.96 MiB | 21.84 MiB/s, done.
Resolving deltas: 100% (52/52), done.
/content/cloth-segmentation
/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1mhF3yqd7R-Uje092eypktNl-RoZNuiCJ

but Gdown can't. Pleas

In [ ]:
# @title Download model
!pip install gdown

import gdown
file_id = '1aS4gB2UucK1twuU5ROg_MYRIxm3jmk1v'
url = f'https://drive.google.com/uc?id={file_id}'

# Download file
output = 'cloth_segm_u2net_latest.pth'
gdown.download(url, output, quiet=False)

print(f'File uploaded as {output}')


Downloading...
From (original): https://drive.google.com/uc?id=1aS4gB2UucK1twuU5ROg_MYRIxm3jmk1v
From (redirected): https://drive.google.com/uc?id=1aS4gB2UucK1twuU5ROg_MYRIxm3jmk1v&confirm=t&uuid=054b3776-2715-4429-9588-aa6f78f87c71
To: /content/cloth-segmentation/cloth_segm_u2net_latest.pth
100%|██████████| 177M/177M [00:01<00:00, 160MB/s]

File uploaded as cloth_segm_u2net_latest.pth


In [ ]:
!cp -r /content/unzipped_folder/* /content/cloth-segmentation/input_images/

In [ ]:
import os
import shutil
from tqdm.notebook import tqdm
from PIL import Image, ImageFile
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import cv2

from data.base_dataset import Normalize_image
from utils.saving_utils import load_checkpoint_mgpu
from networks import U2NET

# Fix truncated image loading
ImageFile.LOAD_TRUNCATED_IMAGES = True

device = 'cuda'

# === Configuration ===
source_image_dir = '/content/unzipped_folder/image'  # <-- Set your input images directory here
image_dir = '/content/cloth-segmentation/input_images'
result_dir = '/content/cloth-segmentation/output_images'
checkpoint_path = '/content/cloth-segmentation/cloth_segm_u2net_latest.pth'

# === Create necessary directories ===
os.makedirs(image_dir, exist_ok=True)
os.makedirs(result_dir, exist_ok=True)

# === Clear previous results ===
for folder in [result_dir, image_dir]:
    for file in os.listdir(folder):
        file_path = os.path.join(folder, file)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            print(f'Failed to delete {file_path}. Reason: {e}')

# === Copy images from source directory ===
for filename in os.listdir(source_image_dir):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        shutil.copy(os.path.join(source_image_dir, filename), os.path.join(image_dir, filename))

# === Color palette for visualization ===
def get_palette(num_cls):
    palette = [0] * (num_cls * 3)
    for j in range(num_cls):
        lab = j
        i = 0
        while lab:
            palette[j * 3 + 0] |= (((lab >> 0) & 1) << (7 - i))
            palette[j * 3 + 1] |= (((lab >> 1) & 1) << (7 - i))
            palette[j * 3 + 2] |= (((lab >> 2) & 1) << (7 - i))
            i += 1
            lab >>= 3
    return palette

# === Image transform ===
transforms_list = [transforms.ToTensor(), Normalize_image(0.5, 0.5)]
transform_rgb = transforms.Compose(transforms_list)

# === Load Model ===
net = U2NET(in_ch=3, out_ch=4)

def load_checkpoint_mgpu(model, checkpoint_path):
    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint file not found: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(checkpoint.get('state_dict', checkpoint), strict=False)
    return model

net = load_checkpoint_mgpu(net, checkpoint_path)
net = net.to(device)
net.eval()

palette = get_palette(4)

# === Process images ===
images_list = sorted(os.listdir(image_dir))
pbar = tqdm(total=len(images_list))

for image_name in images_list:
    img_path = os.path.join(image_dir, image_name)

    try:
        img = Image.open(img_path).convert('RGB')
    except Exception as e:
        print(f"Skipping {image_name}: {e}")
        continue

    original_size = img.size

    # Resize, transform and forward pass
    img_resized = img.resize((768, 768), Image.BICUBIC)
    image_tensor = transform_rgb(img_resized).unsqueeze(0).to(device)
    output_tensor = net(image_tensor)[0]  # First output of U2NET

    # Softmax and get class mask
    output_tensor = F.log_softmax(output_tensor, dim=1)
    output_tensor = torch.max(output_tensor, dim=1, keepdim=True)[1].squeeze().cpu().numpy()

    # Convert mask to binary format
    output_arr = np.where(output_tensor != 0, 255, 0).astype(np.uint8)

    # Dilate mask to expand boundaries
    kernel = np.ones((7, 7), np.uint8)
    output_arr = cv2.dilate(output_arr, kernel, iterations=2)

    # Save output image using the same name
    output_img = Image.fromarray(output_arr, mode='L')
    output_img = output_img.resize(original_size, Image.BICUBIC)
    output_img.save(os.path.join(result_dir, image_name))

    # Optional visualization
    plt.imshow(output_img, cmap='gray')
    plt.title(f'Mask: {image_name}')
    plt.axis('off')
    plt.show()

    pbar.update(1)

pbar.close()


In [ ]:
import shutil
from google.colab import files

# 1. Zip the folder
shutil.make_archive('/content/output_images', 'zip', '/content/cloth-segmentation/output_images')

# 2. Download the zipped folder
files.download('/content/output_images.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

folder_path = '/content/cloth'  # Replace with your folder path

file_names = os.listdir(folder_path)
for name in file_names:
    print(name)


In [ ]:
import shutil
from google.colab import files

# 1. Zip the folder
shutil.make_archive('/content/cloths', 'zip', '/content/cloth')

# 2. Download the zipped folder
files.download('/content/cloths.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!cp -r "/content/cloths.zip" /content/drive/MyDrive/NewDataset

In [ ]:
import cv2
import numpy as np
import os

def extract_cloth(original_img_path, mask_img_path, output_path, mask_threshold=127):
    """
    Extract cloth from the original image using the segmentation mask.

    Parameters:
    - original_img_path: Path to the original input image.
    - mask_img_path: Path to the segmentation mask image (grayscale or multi-channel).
    - output_path: Path to save the extracted cloth image with transparent background.
    - mask_threshold: Threshold to binarize the mask (default 127).
    """

    # Load original image
    original = cv2.imread(original_img_path, cv2.IMREAD_UNCHANGED)
    if original is None:
        print(f"Error: Could not load original image {original_img_path}")
        return

    # Load mask image (assuming grayscale mask)
    mask = cv2.imread(mask_img_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        print(f"Error: Could not load mask image {mask_img_path}")
        return

    # Binarize the mask
    _, binary_mask = cv2.threshold(mask, mask_threshold, 255, cv2.THRESH_BINARY)

    # Check number of channels
    if original.ndim != 3 or original.shape[2] not in [3, 4]:
        print(f"Skipping {original_img_path}: unsupported number of channels ({original.shape})")
        return

    # Split channels
    if original.shape[2] == 4:
        b, g, r, _ = cv2.split(original)  # discard existing alpha
    else:
        b, g, r = cv2.split(original)

    # Use mask as new alpha channel
    alpha = binary_mask
    rgba = cv2.merge([b, g, r, alpha])

    # Save the output image
    cv2.imwrite(output_path, rgba)
    print(f"✅ Extracted cloth saved to: {output_path}")

# === Example usage ===
if __name__ == "__main__":
    original_images_folder = "/content/images_folder/image"
    masks_folder = "/content/cloth_segmentation_folder/Cloth Segmentation/"
    output_folder = "/content/cloth"

    os.makedirs(output_folder, exist_ok=True)

    success_count = 0
    failure_count = 0

    for filename in os.listdir(original_images_folder):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            original_img_path = os.path.join(original_images_folder, filename)
            mask_img_path = os.path.join(masks_folder, filename)  # assumes same name for mask
            output_name = os.path.splitext(filename)[0] + '.png'
            output_path = os.path.join(output_folder, output_name)

            try:
                extract_cloth(original_img_path, mask_img_path, output_path)
                success_count += 1
            except Exception as e:
                print(f"❌ Skipping {filename} due to error: {e}")
                failure_count += 1

    print(f"\n=== Process complete ===\nSuccess: {success_count}, Failed: {failure_count}")


✅ Extracted cloth saved to: /content/cloth/00283_00.png
✅ Extracted cloth saved to: /content/cloth/00499_00.png
✅ Extracted cloth saved to: /content/cloth/02913_00.png
✅ Extracted cloth saved to: /content/cloth/01595_00.png
✅ Extracted cloth saved to: /content/cloth/01012_00.png
✅ Extracted cloth saved to: /content/cloth/00591_00.png
✅ Extracted cloth saved to: /content/cloth/02199_00.png
✅ Extracted cloth saved to: /content/cloth/02780_00.png
✅ Extracted cloth saved to: /content/cloth/01651_00.png
✅ Extracted cloth saved to: /content/cloth/02381_00.png
✅ Extracted cloth saved to: /content/cloth/03030_00.png
✅ Extracted cloth saved to: /content/cloth/02979_00.png
✅ Extracted cloth saved to: /content/cloth/02665_00.png
✅ Extracted cloth saved to: /content/cloth/00338_00.png
✅ Extracted cloth saved to: /content/cloth/02283_00.png
✅ Extracted cloth saved to: /content/cloth/02060_00.png
✅ Extracted cloth saved to: /content/cloth/02966_00.png
✅ Extracted cloth saved to: /content/cloth/02850

In [ ]:
# @title Download results from output_images

import os
from google.colab import files

# Function to download files
def download_files_from_folder(folder_path):
    # Get the list of all files in the folder
    files_list = os.listdir(folder_path)

    # Filter files, keeping only images
    image_files = [f for f in files_list if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]

    for image_file in image_files:
        # Path to the image
        file_path = os.path.join(folder_path, image_file)
        # Download the file
        files.download(file_path)

# Specify the path to the folder with images
folder_path = 'output_images'
download_files_from_folder(folder_path)